# Multi-Hop Retrieval — Evaluate FakeEncoder (retrieval_v2)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_dev.jsonl`
   - `musique_ans_v1.0_train.jsonl`
   - `fakencoder_best.pt`  ← trained in train_fakencoder_kaggle.ipynb
   - `model2_best.pt`  ← (optional) trained in train_model2_kaggle.ipynb

**What this runs (3 systems compared):**
- `MDR (dense, iterative)` — published baseline
- `Graph + direct cosine (FE)` — FE passage embeddings, no complement (Model 1 query enc)
- `FULL: FE graph + complement` — complement edge vectors + Model 2 query enc (if available)

**Key diagnostic:**
- `FULL R@10 > Graph+cos R@10` by >1%: complement helps, Model 2 alignment works
- `FULL ≈ Graph+cos`: FakeEncoder collapsed or Model 2 not yet trained

In [ ]:
# ── [EDIT IF NEEDED] ───────────────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v2"
MAX_EXAMPLES    = 300    # change to None for all 2417 dev queries
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")

cc = torch.cuda.get_device_properties(0).major
if cc < 7:
    raise RuntimeError(
        f"GPU is P100 (sm_60) — not supported.\n"
        "FIX: Settings → Accelerator → change to T4 GPU"
    )

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone / pull repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' 'sentence-transformers>=2.2.0' gdown faiss-cpu")
print("Dependencies ready")

In [ ]:
# 3. Download everything from Google Drive (data + checkpoint)
import os, gdown

download_dir = "/kaggle/working/drive_data"

if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}") / 1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Set up file paths (symlinks + checkpoint copy)
import os, shutil

drive = "/kaggle/working/drive_data"

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models",       exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",        exist_ok=True)
os.makedirs(f"{WORK_DIR}/results",      exist_ok=True)

# MuSiQue JSONL -> symlinks
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

# FakeEncoder checkpoint (Model 1) — required
ckpt = "fakencoder_best.pt"
src  = f"{drive}/{ckpt}"
dst  = f"{WORK_DIR}/models/{ckpt}"
if os.path.exists(src):
    if not os.path.exists(dst):
        print(f"  Copying {ckpt} ({os.path.getsize(src)/1e6:.0f} MB)...", flush=True)
        shutil.copy(src, dst)
    print(f"  {ckpt}: OK ({os.path.getsize(dst)/1e6:.0f} MB)")
else:
    print(f"  WARNING: {ckpt} not found in Drive — will run MDR-only")

# QueryEncoder checkpoint (Model 2) — optional, enables better query alignment
ckpt2 = "model2_best.pt"
src2  = f"{drive}/{ckpt2}"
dst2  = f"{WORK_DIR}/models/{ckpt2}"
if os.path.exists(src2):
    if not os.path.exists(dst2):
        print(f"  Copying {ckpt2} ({os.path.getsize(src2)/1e6:.0f} MB)...", flush=True)
        shutil.copy(src2, dst2)
    print(f"  {ckpt2}: OK ({os.path.getsize(dst2)/1e6:.0f} MB)")
else:
    print(f"  {ckpt2}: not in Drive — FULL system will use Model 1 for query encoding")

print("\nAll paths ready")

---
## Run Full Evaluation

Calls `main()` directly from the eval script — no subprocess, no `__file__` issues.

In [ ]:
# 5. Run evaluation
import sys, os
os.chdir(WORK_DIR)

# Add retrieval_v2/ and retrieval/ to path so imports resolve correctly
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/../retrieval")

from run_eval import main
results = main(max_examples=MAX_EXAMPLES)

In [ ]:
# 6. Summary table
print("\n=== FINAL RESULTS ===")
for name, metrics in results.items():
    r10 = metrics.get('recall@10', 0)
    r5  = metrics.get('recall@5',  0)
    r2  = metrics.get('recall@2',  0)
    print(f"  {name:45s}  R@2={r2:.3f}  R@5={r5:.3f}  R@10={r10:.3f}")